In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

# Step 1: Define the structure of the Bayesian Network
model = DiscreteBayesianNetwork([
    ('Weather', 'Traffic'),
    ('Traffic', 'LateToWork')
])

# Step 2: Define the CPDs with state names

# Weather: sunny (0), rainy (1)
cpd_weather = TabularCPD(
    variable='Weather', variable_card=2,
    values=[[0.6], [0.4]],
    state_names={'Weather': ['sunny', 'rainy']}
)

# Traffic depends on Weather
# Traffic: light (0), heavy (1)
cpd_traffic = TabularCPD(
    variable='Traffic', variable_card=2,
    values=[
        #light #heavy
        [0.8, 0.3],
        [0.2, 0.7]

    ],
    evidence=['Weather'], evidence_card=[2],
    state_names={
        'Traffic': ['light', 'heavy'],
        'Weather': ['sunny', 'rainy']
    }
)

# LateToWork depends on Traffic
# LateToWork: yes (0), no (1)
cpd_late = TabularCPD(
    variable='LateToWork', variable_card=2,
    values=[
        [0.2, 0.9],  # yes
        [0.8, 0.1]   # no
    ],
    evidence=['Traffic'], evidence_card=[2],
    state_names={
        'LateToWork': ['yes', 'no'],
        'Traffic': ['light', 'heavy']
    }
)

# Step 3: Add CPDs to the model
model.add_cpds(cpd_weather, cpd_traffic, cpd_late)

# Step 4: Check if the model is valid
assert model.check_model(), "Model structure or CPDs are invalid."

# Step 5: Inference Engine
inference = VariableElimination(model)

# Step 6: Example Inferences
print("\n1. P(LateToWork)")
print(inference.query(variables=['LateToWork']))

print("\n2. P(LateToWork | Traffic=heavy)")
print(inference.query(variables=['LateToWork'], evidence={'Traffic': 'heavy'}))

print("\n3. P(LateToWork | Weather=rainy)")
print(inference.query(variables=['LateToWork'], evidence={'Weather': 'rainy'}))

print("\n4. P(Weather | LateToWork=yes)")
print(inference.query(variables=['Weather'], evidence={'LateToWork': 'yes'}))
